# Build a RAG application with Azure Cosmos DB and LangChain

Think of this notebook as a build-along session. At each step, I'll call out what we're doing, why it matters, and what to expect before you run the next cell.

We're building a practical architecture where one data layer supports multiple AI app needs:

1. Vector retrieval for RAG
2. Full-text and hybrid search for stronger recall
3. Semantic cache to reduce cost and latency
4. Chat history for conversational continuity

By the end, running every cell launches an interactive chat app powered entirely by Azure Cosmos DB and Azure OpenAI.

Next step, let's quickly align on what RAG is and why teams use it in production.

## 0. What is Retrieval-Augmented Generation (RAG)?

Imagine you are building a support assistant and a user asks: *"Can I return an item I purchased 45 days ago?"* A model can answer confidently, but without retrieval it may still be wrong because it does not know your latest policy docs.

RAG solves this by adding one crucial move before generation: retrieve trusted context first, then answer. That keeps responses grounded in your actual data instead of model guesswork.

In real teams, this is the difference between a demo that sounds good and a system people can trust.

Next step, we'll install the dependencies we need before we start coding.

## 1. Install Dependencies

This installs everything we need for data access, model calls, retrieval, and the demo UI.

If your environment is already prepared you can skip this, but otherwise run it now so we don't hit import issues later.

To confirm the package version, release notes, and install source, see [langchain-azure-cosmosdb on PyPI](https://pypi.org/project/langchain-azure-cosmosdb/).

Next step is importing the exact classes we will use in the pipeline.


In [2]:
%pip install -qU azure-cosmos azure-identity python-dotenv pypdf gradio
%pip install -qU langchain langchain-core langchain-openai langchain-text-splitters langchain-community
%pip install -qU langchain-azure-cosmosdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

Here we bring in the building blocks for each stage: load documents, split text, create embeddings, and store/retrieve from Cosmos DB.

Next step, we'll enter the credentials so the notebook can reach your Azure resources.

In [3]:
# Standard library imports
import os
from pathlib import Path

# Environment variable loading
from dotenv import dotenv_values

# Azure Cosmos DB client and partitioning
from azure.cosmos import CosmosClient
from azure.cosmos.partition_key import PartitionKey
from azure.identity import DefaultAzureCredential

# LangChain model and text processing utilities
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.globals import set_llm_cache

# Cosmos DB integrations for vector search, semantic cache, and chat history
from langchain_azure_cosmosdb import (
    AzureCosmosDBNoSqlVectorSearch,
    AzureCosmosDBNoSqlSemanticCache,
    CosmosDBChatMessageHistory,
)

# Lightweight UI for interactive testing
import gradio as gr

C:\Users\t-saukapoor\AppData\Local\Temp\ipykernel_30364\2290081868.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. Configuration

To keep your secrets out of this notebook, and out of any GitHub repo, we load all Azure credentials and resource names from a local `.env` file instead of hardcoding them. Because the keys live in a separate file that you add to `.gitignore`, you can't accidentally publish them when you commit the `.ipynb`.

**One-time setup:**

1. Open the provided `.env_template` file (in the same folder as this notebook) and replace each placeholder with your real values.
2. **Rename `.env_template` to `.env`**, since the code below reads from `.env` and that rename is what actually makes your keys load.
3. Add `.env` to your `.gitignore` so it is never committed.

The `.env_template` already contains every key the notebook needs:
```env
cosmos_endpoint=https://<account>.documents.azure.com:443/
cosmos_key=<your-cosmos-key-or-leave-empty-for-managed-identity>
cosmos_database_name=rag_agents_db
cosmos_vector_container_name=documents
cosmos_history_container_name=chat_history
cosmos_cache_container_name=semantic_cache

openai_endpoint=https://<aoai-resource>.openai.azure.com/
openai_key=<your-openai-api-key>
openai_version=2024-02-15-preview
openai_embeddings_deployment=text-embedding-3-small
openai_embeddings_model=text-embedding-3-small
openai_embeddings_dimensions=1536
openai_completions_deployment=gpt-4o
```

The next cell reads this file with `dotenv_values(".env")` **and** connects to Cosmos DB in one step, so everything downstream is ready to go.

> **Security tip:** In production, prefer **Azure Managed Identity** (`DefaultAzureCredential()`) over keys by simply leaving `cosmos_key` empty.

### Where to set up these resources

- [Create an Azure Cosmos DB account](https://learn.microsoft.com/azure/cosmos-db/create-cosmosdb-resources-portal), where you'll find `cosmos_endpoint` and `cosmos_key` under **Keys**.
- [Create an Azure OpenAI resource](https://learn.microsoft.com/azure/ai-services/openai/how-to/create-resource), where you'll find `openai_endpoint` and `openai_key` under **Keys and Endpoint** before deploying the embedding and chat models to get their deployment names.
- [LangChain × Azure Cosmos DB guide](https://docs.langchain.com/oss/python/integrations/vectorstores/azure_cosmos_db_no_sql), which is a handy reference for vector store config, indexing policies, and search modes.

Next step, we'll initialize the embedding and chat models.


In [ ]:
# ============ LOAD CONFIGURATION FROM .env ============
# Secrets live in .env (not in this notebook), so they won't be committed to
# source control. If you haven't already: fill in .env_template, rename it to
# .env, and add .env to your .gitignore.
env_path = Path.cwd() / ".env"

# Make sure the .env file actually exists where we're looking for it
if not env_path.exists():
    hint = ""
    if (Path.cwd() / ".env_template").exists():
            hint = " You still have '.env_template' here, so fill it in and rename it to '.env'."
    raise FileNotFoundError(f"No .env file found at: {env_path}.{hint}")

config = dotenv_values(env_path)

# Make sure every key the notebook needs is present and non-empty.
# (cosmos_key is intentionally optional, so leave it blank to use Managed Identity.)
required_keys = [
    "cosmos_endpoint",
    "cosmos_database_name",
    "cosmos_vector_container_name",
    "cosmos_history_container_name",
    "cosmos_cache_container_name",
    "openai_endpoint",
    "openai_key",
    "openai_version",
    "openai_embeddings_deployment",
    "openai_embeddings_model",
    "openai_embeddings_dimensions",
    "openai_completions_deployment",
]
missing = [key for key in required_keys if not config.get(key)]
if missing:
    raise KeyError(
        f"Your .env is missing or has empty values for: {missing}. "
        "Compare it against .env_template and make sure each line is filled in."
    )

# Cosmos DB settings
cosmos_endpoint = config["cosmos_endpoint"]
cosmos_key = config.get("cosmos_key", "")
cosmos_database = config["cosmos_database_name"]
cosmos_vector_container = config["cosmos_vector_container_name"]
cosmos_history_container = config["cosmos_history_container_name"]
cosmos_cache_container = config["cosmos_cache_container_name"]

# Azure OpenAI settings
openai_endpoint = config["openai_endpoint"]
openai_key = config["openai_key"]
openai_api_version = config["openai_version"]
openai_embeddings_deployment = config["openai_embeddings_deployment"]
openai_embeddings_model = config["openai_embeddings_model"]
openai_embeddings_dimensions = int(config["openai_embeddings_dimensions"])
openai_completions_deployment = config["openai_completions_deployment"]

# ============ INITIALIZE CONNECTIONS ============
# Use the key if provided, otherwise fall back to Azure Managed Identity
cosmos_credential = cosmos_key if cosmos_key else DefaultAzureCredential()
cosmos_client = CosmosClient(cosmos_endpoint, credential=cosmos_credential)

# Suppress OpenAI's global API key check (we're using the Azure endpoint instead)
os.environ["OPENAI_API_KEY"] = ""

print("✅ Configuration loaded from .env and Cosmos DB connected. You're ready to run the rest of the notebook.")


✅ Configuration loaded from .env and Cosmos DB connected. You're ready to run the rest of the notebook.


## 4. Azure OpenAI Models

We set up two model endpoints, each with a different job:

- Embeddings model for retrieval
- Chat model for final response generation

Keeping these separate gives you better control when tuning quality, latency, and cost.

Next step, we'll load the source PDFs that will become your knowledge base.

In [5]:
# Embedding model converts text chunks into vectors for retrieval
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=openai_endpoint,
    api_key=openai_key,
    api_version=openai_api_version,
    azure_deployment=openai_embeddings_deployment,
    model=openai_embeddings_model,
    dimensions=openai_embeddings_dimensions,
)

# Chat model generates grounded answers from retrieved context
llm = AzureChatOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_key,
    api_version=openai_api_version,
    azure_deployment=openai_completions_deployment,
    temperature=0.2,
)

## 5. Load Documents

Now we pull in the raw knowledge source from PDFs and convert it into LangChain documents.

**How this works:** the cell below automatically loads **every `.pdf` file sitting in the `PDF_Documents` folder next to this notebook**. Just drop your PDFs into that folder, with no configuration needed. To use a different folder instead, change `pdf_dir`.

**Need sample data?** You can add your own PDFs, or run the `download_pdf.py` script inside the `PDF_Documents` folder to fetch a set of sample PDFs to use (from that folder, run `python download_pdf.py`).

To understand loader behavior and alternatives for other file types, see the [PyPDFLoader docs](https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/).

Next step, we'll split those pages into chunk sizes that retrieval can actually work with.


In [6]:
# Load every PDF found in the PDF_Documents folder next to this notebook
pdf_dir = Path.cwd() / "PDF_Documents"
docs = []

# Find and load each PDF page into LangChain Document objects
pdf_files = sorted(pdf_dir.glob("*.pdf")) if pdf_dir.exists() else []
print(f"Found {len(pdf_files)} PDF file(s) in {pdf_dir}:")
for pdf_path in pdf_files:
    print(f"  - {pdf_path.name}")
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())

# Quick sanity checks so we can confirm loading worked
print(f"\nLoaded {len(docs)} document pages")
if docs:
    print(docs[0].page_content[:500])
else:
    print("No PDFs found. Place one or more .pdf files in the 'PDF_Documents' folder next to this notebook, then re-run.")

Found 5 PDF file(s) in c:\Users\t-saukapoor\Downloads\RAG_Application_Example_CosmosDB\PDF_Documents:
  - ancient_egyptian_costumes.pdf
  - ancient_history_volume_1.pdf
  - ancient_rome_history.pdf
  - egyptian_greek_looms.pdf
  - egyptian_immortality.pdf

Loaded 575 document pages



## 6. Split Documents into Chunks

Now we break long pages into retrievable units.

If the chunks are too big retrieval gets noisy, and if they are too small the context gets fragmented, so the values below are good starter defaults for a tutorial.

To learn how to tune `chunk_size` and `chunk_overlap` for your document style and query behavior, see [LangChain text splitter concepts](https://python.langchain.com/docs/concepts/text_splitters/).

Next step, we'll define the indexing policies so Cosmos DB can support both semantic and hybrid retrieval.


In [7]:
# Split pages into overlapping chunks so retrieval has focused but connected context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
)

# These chunks are what get embedded and indexed
documents = text_splitter.split_documents(docs)
print(f"Created {len(documents)} chunks")

Created 1719 chunks


## 7. Define Cosmos DB Indexing Policies

Indexes power fast retrieval.

We set the schema and policies Cosmos DB needs: a vector embedding policy (diskANN, cosine), a vector index, and a full-text index for lexical search.

To compare the vector index types (diskANN, quantizedFlat, flat) and their tradeoffs for your data size, see the [Cosmos DB vector index guide](https://learn.microsoft.com/azure/cosmos-db/vector-search). To learn how BM25 ranking powers lexical retrieval, see [Cosmos DB full-text search](https://learn.microsoft.com/azure/cosmos-db/gen-ai/full-text-search).

Next step, we'll wrap a Cosmos DB container with LangChain's vector store. This also creates the database and container for us.


In [8]:
# Define vector embedding schema stored in Cosmos DB
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embedding",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": openai_embeddings_dimensions,
        }
    ]
}

# Enable vector + full-text indexing for hybrid retrieval
indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [
        {"path": '/"_etag"/?'},
    ],
    "vectorIndexes": [
        {
            "path": "/embedding",
            "type": "diskANN",
        }
    ],
    "fullTextIndexes": [
        {
            "path": "/text",
        }
    ],
}

# Full-text policy: declares which paths are searchable and their language
full_text_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [
        {
            "path": "/text",
            "language": "en-US",
        }
    ],
}

# Keep partitioning simple for tutorial data insertion
cosmos_container_properties = {
    "partition_key": PartitionKey(path="/id")
}

cosmos_database_properties = {
    "id": cosmos_database
}

# Map document content and embedding field names used by the vector store
vector_search_fields = {
    "text_field": "text",
    "embedding_field": "embedding",
}

## 8. Create Vector Store

Now we wrap a Cosmos DB container with LangChain's `AzureCosmosDBNoSqlVectorSearch`. Running the cell below creates the database and `documents` container (using the policies from the previous step) and gives us a single object that handles both vector and hybrid retrieval.

To explore the search parameters, filters, and hybrid ranking options, see the [LangChain vector store guide](https://python.langchain.com/docs/integrations/vectorstores/azure_cosmos_db_no_sql).

Next step, we'll embed and load our chunks into it.


In [9]:
# Wrap Cosmos DB container in LangChain vector store interface
vectorstore = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client=cosmos_client,
    embedding=embeddings,
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=indexing_policy,
    full_text_policy=full_text_policy,
    cosmos_container_properties=cosmos_container_properties,
    cosmos_database_properties=cosmos_database_properties,
    vector_search_fields=vector_search_fields,
    database_name=cosmos_database,
    container_name=cosmos_vector_container,
    full_text_search_enabled=True,
)

## 9. Ingest Documents into Vector Store

We embed each chunk and write it to Cosmos DB, where it's indexed for both vector and full-text search.

> **Avoid throttling:** ingestion speed is bound by your container's throughput. Provision **at least 1000 RU/s** (or enable Autoscale) on the `documents` container so writes don't hit HTTP 429 (TooManyRequests) errors.

Next step, we'll query the indexed data using pure vector similarity.


In [10]:
# Embed each chunk and insert it into Cosmos DB, in small batches.
if documents:
    batch_size = 100
    total = len(documents)

    for start in range(0, total, batch_size):
        batch = documents[start:start + batch_size]
        vectorstore.add_documents(batch)
        print(f"Inserted {start + len(batch)}/{total} chunks")

    print("\n✅ Done.")
else:
    print("No documents to insert")


Inserted 100/1719 chunks
Inserted 200/1719 chunks
Inserted 300/1719 chunks
Inserted 400/1719 chunks
Inserted 500/1719 chunks
Inserted 600/1719 chunks
Inserted 700/1719 chunks
Inserted 800/1719 chunks
Inserted 900/1719 chunks
Inserted 1000/1719 chunks
Inserted 1100/1719 chunks
Inserted 1200/1719 chunks
Inserted 1300/1719 chunks
Inserted 1400/1719 chunks
Inserted 1500/1719 chunks
Inserted 1600/1719 chunks
Inserted 1700/1719 chunks
Inserted 1719/1719 chunks

✅ Done.


## 10. Vector Search

Find the most relevant chunks for a question using cosine similarity: we embed the query with the same model, then retrieve the top-k closest vectors from Cosmos DB.

This is pure semantic retrieval, so it catches meaning even when the exact words don't line up.

Next step, we'll combine vectors with full-text ranking for even better results.

In [11]:
# Vector (semantic) search: find chunks closest in meaning to the question
query = "What did the ancient Egyptians believe about the afterlife?"
results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results, start=1):
    print(f"--- Vector result {i} ---")
    print(doc.page_content[:700])
    print()


--- Vector result 1 ---
destroyed or overthrown; but such vague expressions afford no certainty as to how far the Egyptians in general believed in
the existence of a hell as a place of punishment or purification for the wicked;44 or whether, as seems more probable, they
held some general belief that when judgment was pronounced against a man his heart and other immortal parts were not
restored to him. For such a man no re-edification and no resurrection was possible. The immortal elements were divine, and
by nature pure and imperishable; but they could be preserved from entering the O     , from re-entering the hull of the man
who had proved himself unworthy of them. The soul, indeed, as such did not die, although 

--- Vector result 2 ---
destroyed or overthrown; but such vague expressions afford no certainty as to how far the Egyptians in general believed in
the existence of a hell as a place of punishment or purification for the wicked;44 or whether, as seems more probable, they
hel

## 11. Hybrid Search (Vector + Full-Text)

Hybrid search blends two signals: vector similarity for semantic relevance, and BM25 full-text ranking for exact keyword matches. Results are merged and re-ranked.

This catches both meaning-based and keyword-based relevance. It is useful for names, IDs, and domain terms that pure vectors can miss.

Next step, we'll wrap retrieval and generation into one reusable RAG function.

In [12]:
# Hybrid search: blend vector similarity with a keyword (BM25) filter
hybrid_query = "ancient Egyptian beliefs about the soul"

hybrid_results = vectorstore.similarity_search(
    hybrid_query,
    k=5,
    search_type="hybrid",
    full_text_rank_filter=[
        {"search_field": "text", "search_text": "soul"},
    ],
)

for i, doc in enumerate(hybrid_results, start=1):
    print(f"--- Hybrid result {i} ---")
    print(doc.page_content[:700])
    print()


--- Hybrid result 1 ---
5
L
THE
ANCIENT EGYPTIAN DOCTRINE 
OF THE 
IMMORTALITY OF THE SOUL.
ITTLE as we know of the ancient Egyptian religion in its entirety, and of its motley mixture of childishly crude fetichism and
deep philosophic thought, of superstition and true religious worship, of polytheism, henotheism, and pantheism, one
dogma stands out clearly from this confusion, one article of belief to which the Egyptian religion owes its unique position
among all other religions of antiquity−-the doctrine of the immortality of the human soul. It is true that other ancient religions
attained to a similar dogma, for the belief was early developed among Semites, Indo-Germanians, Turanians, and
Mongolians; but in all 

--- Hybrid result 2 ---
5
L
THE
ANCIENT EGYPTIAN DOCTRINE 
OF THE 
IMMORTALITY OF THE SOUL.
ITTLE as we know of the ancient Egyptian religion in its entirety, and of its motley mixture of childishly crude fetichism and
deep philosophic thought, of superstition and true reli

## 12. Build a RAG Function

Now we turn separate steps into one reusable flow:

1. Retrieve context
2. Build a grounded prompt
3. Generate an answer
4. Return answer plus sources

This is where your notebook shifts from retrieval experiments to an actual application pattern.

Next step, we'll run a quick end-to-end test.

In [13]:
# System instruction forces grounded answers only
SYSTEM_PROMPT = """
You are a helpful assistant. Answer the user's question using only the retrieved context.
If the context does not contain enough information, say you do not know.
Do not invent facts that are not supported by the retrieved context.
"""


def format_docs(docs):
    # Label each chunk so source ordering is visible in the prompt
    return "\n\n".join(
        f"Source {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )


def retrieve_context(question: str, k: int = 5, mode: str = "hybrid"):
    # Default to hybrid retrieval to balance semantic relevance and lexical precision
    if mode == "hybrid":
        return vectorstore.similarity_search(
            question,
            k=k,
            search_type="hybrid",
            full_text_rank_filter=[
                {"search_field": "text", "search_text": question}
            ],
        )
    return vectorstore.similarity_search(question, k=k)


def ask_rag(question: str, mode: str = "hybrid"):
    # 1) Retrieve supporting evidence
    retrieved_docs = retrieve_context(question, mode=mode)
    context = format_docs(retrieved_docs)

    # 2) Build user prompt that includes retrieved context
    user_prompt = f"""
Retrieved context:
{context}

Question:
{question}
"""

    # 3) Ask model for grounded response
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    # Return both answer and sources for debugging/evaluation
    return response.content, retrieved_docs

## 13. Test RAG

This is your first full pipeline check.

Look at both the answer and the retrieved sources. If the answer feels weak, retrieval quality is usually the first place to tune.

Next step, we'll add semantic caching to reduce repeated LLM calls.

In [14]:
# End-to-end RAG smoke test
answer, sources = ask_rag("What does the text say about ancient Egyptian beliefs about the soul?", mode="hybrid")
print(answer)
print("\nNumber of retrieved sources:", len(sources))

The text says that, despite the overall complexity and mixed character of ancient Egyptian religion (ranging from “crude fetichism” to “deep philosophic thought”), one belief stands out clearly: **the doctrine of the immortality of the human soul**.

It emphasizes that Egypt is distinctive because it held **one of the most elaborated forms of belief in immortality** alongside **very elementary conceptions of higher beings**. It also notes that Egyptian eschatology shows that **a whole nation believed in the soul’s immortality about four thousand years before the birth of Christ**, and that they had already **formed a fairly clear picture of the future life**, even if it can seem strange to modern readers.

Number of retrieved sources: 5


## 14. Add Semantic Cache

When a new prompt is semantically close to a previous one, the system can reuse an earlier response instead of calling the LLM again. That means lower cost and faster responses for repeated intent.

For other caching backends and configuration options, see the [LangChain LLM caching guide](https://python.langchain.com/docs/how_to/llm_caching/).

Next step, we'll add short-term chat memory for multi-turn conversations.


In [15]:
# The semantic cache only needs vector search, so we give it its own indexing
# policy WITHOUT the full-text index (avoids a full-text policy mismatch on the
# cache container).
cache_indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [
        {"path": '/"_etag"/?'},
    ],
    "vectorIndexes": [
        {
            "path": "/embedding",
            "type": "diskANN",
        }
    ],
}
cache_vector_embedding_policy = vector_embedding_policy
cache_container_properties = cosmos_container_properties

# Cache stores semantically similar prompt/response pairs
semantic_cache = AzureCosmosDBNoSqlSemanticCache(
    cosmos_client=cosmos_client,
    embedding=embeddings,
    vector_embedding_policy=cache_vector_embedding_policy,
    indexing_policy=cache_indexing_policy,
    cosmos_container_properties=cache_container_properties,
    cosmos_database_properties={"id": cosmos_database},
    vector_search_fields=vector_search_fields,
    database_name=cosmos_database,
    container_name=cosmos_cache_container,
    score_threshold=0.9,
)

# Register as global LangChain cache
set_llm_cache(semantic_cache)
print("Semantic cache configured")

Semantic cache configured


## 15. Chat History

To hold a real conversation, the assistant needs to remember earlier turns. We use `CosmosDBChatMessageHistory` to persist user and AI messages in a Cosmos DB container, **keyed by `user_id`**. Each user gets their own transcript inside the *same* container. The `get_history(user_id)` factory below returns (and caches) one history object per user, which is what powers the multi-user demo at the end.

Next step, we'll combine retrieval with this history into one conversational function.


In [16]:
# We key chat history by user_id so each user gets their OWN persisted transcript
# inside the SAME Cosmos DB container. Switching users => different history.

# Two demo profiles we'll show side-by-side in the multi-user UI later
DEMO_USERS = ["James", "Saurish"]

# Cache one history object per user so we reuse the same Cosmos connection per user
_history_cache = {}


def get_history(user_id: str) -> CosmosDBChatMessageHistory:
    """Return a per-user CosmosDBChatMessageHistory, creating it on first use."""
    if user_id not in _history_cache:
        history = CosmosDBChatMessageHistory(
            cosmos_endpoint=cosmos_endpoint,
            cosmos_database=cosmos_database,
            cosmos_container=cosmos_history_container,
            credential=cosmos_credential,
            session_id=f"session-{user_id}",  # one session per user for this demo
            user_id=user_id,
        )
        # Create the underlying container if it doesn't exist yet (safe to repeat)
        history.prepare_cosmos()
        _history_cache[user_id] = history
    return _history_cache[user_id]


# Warm up the demo users so their containers/partitions exist up front
for _user in DEMO_USERS:
    get_history(_user)

print(f"Chat history ready (keyed by user_id) for: {', '.join(DEMO_USERS)}")


Chat history ready (keyed by user_id) for: James, Saurish


## 16. Conversational RAG

Now we combine retrieval with chat history. This function:

1. Retrieves context for the current question.
2. Pulls this user's last few messages from chat history.
3. Builds a prompt with history + context + question.
4. Generates a response.
5. Saves the new turn back to chat history.

The result is a grounded, memory-aware assistant. Before each call we also check the semantic cache: if a semantically similar question was already answered, the stored answer is reused and tagged with a **Cached response** badge, so you can see when no new model call was needed.

Next step, we'll wrap it in a simple web UI.


In [17]:
# Before calling the model, check whether the cache already has an answer for
# this prompt. LangChain keys the cache on the prompt + model settings, so we
# rebuild that same key to look it up.
from langchain_core.load.dump import dumps


def _was_served_from_cache(messages) -> bool:
    try:
        return semantic_cache.lookup(dumps(messages), llm._get_llm_string()) is not None
    except Exception:
        return False


def ask_conversational_rag(question: str, user_id: str, mode: str = "hybrid"):
    # Each user has their own history, persisted under their user_id
    history = get_history(user_id)

    # 1) Retrieve relevant context for the question
    retrieved_docs = retrieve_context(question, mode=mode)
    context = format_docs(retrieved_docs)

    # 2) Pull this user's recent turns so follow-ups stay coherent
    recent_messages = history.messages[-6:]
    history_text = "\n".join(f"{m.type}: {m.content}" for m in recent_messages)

    # 3) Build the prompt: history + retrieved context + question
    user_prompt = f"""
Recent conversation:
{history_text}

Retrieved context:
{context}

Question:
{question}
"""

    # 4) Check the cache, then generate (a cache hit skips the model call)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ]
    served_from_cache = _was_served_from_cache(messages)
    response = llm.invoke(messages)

    # 5) Persist this turn under this user's history
    history.add_user_message(question)
    history.add_ai_message(response.content)

    # 6) Tag answers that were reused from the cache
    answer = response.content
    if served_from_cache:
        answer += "\n\n> 🟢 **Cached response** (reused from the semantic cache)"

    return answer, retrieved_docs


## 17. Interactive Chat UI with Gradio

We use Gradio to put a chat interface in front of the conversational RAG pipeline. The app has a single chat view with a **profile dropdown** (search and select between users such as **James** and **Saurish**), running as one app against one chat-history container; only the `user_id` differs. Selecting a profile preloads that user's stored history, so you can chat as one user, switch profiles in the same workflow, and watch the conversations stay separate (no overlap). Because the history is stored in the database, it also persists across a re-run (use **🔄 Reload from Cosmos DB** to load it again).

For more ways to customize the chat UI, see the [Gradio chat interface guide](https://www.gradio.app/guides/creating-a-chatbot-fast).

Next step, we'll launch the app.


In [19]:
def load_history(user_id: str):
    # Load a user's stored turns from Cosmos DB into Gradio "messages" format:
    # [{"role": ..., "content": ...}, ...]. This is what the chat shows on open.
    history = get_history(user_id)
    messages = []
    for msg in history.messages:
        role = "user" if msg.type == "human" else "assistant"
        messages.append({"role": role, "content": msg.content})
    return messages


# Single-tab UI: one app and one chat-history container, with a dropdown to pick
# the active user. Selecting a different user reloads that user's transcript, so
# you can switch profiles in the same workflow and watch the histories stay
# separate (no overlap), all keyed by user_id.
with gr.Blocks(fill_width=True, title="Cosmos DB RAG: Per-User Chat History") as demo:
    gr.Markdown(
        "# Cosmos DB RAG Chatbot: Per-User Chat History\n"
        "One app against one chat-history container; only the `user_id` differs, "
        "so each user sees their own transcript. Use the **profile dropdown** to "
        "switch between users in the same workflow and confirm the histories don't "
        "overlap. They also persist across a re-run because they're stored in the "
        "database (hit **🔄 Reload** to load again)."
    )

    user_dropdown = gr.Dropdown(
        choices=DEMO_USERS,
        value=DEMO_USERS[0],
        label="👤 User profile",
        info="Search and select a profile to load that user's chat history.",
    )
    header = gr.Markdown(f"### 👤 Chatting as **{DEMO_USERS[0]}**")
    # Gradio 6 uses the "messages" format (list of {"role", "content"} dicts) by
    # default, so no `type=` argument is needed.
    chatbot = gr.Chatbot(height=420, value=load_history(DEMO_USERS[0]))  # preload first user
    msg = gr.Textbox(placeholder=f"Ask a question as {DEMO_USERS[0]}...", show_label=False)
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        reload_btn = gr.Button("🔄 Reload from Cosmos DB")
    gr.Markdown(
        "_Reload re-reads the selected user's saved conversation from the Cosmos DB "
        "`chat_history` container, so you can confirm the messages are stored in "
        "the database rather than only shown on screen._"
    )

    def respond(user_message, chat_messages, user_id):
        # user_id comes from the dropdown, so every turn reads/writes only that
        # user's history.
        if not user_message:
            return chat_messages, ""
        answer, _sources = ask_conversational_rag(user_message, user_id=user_id, mode="hybrid")
        chat_messages = chat_messages + [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer},
        ]
        return chat_messages, ""

    def switch_user(user_id):
        # On profile change, swap the header, the placeholder, and reload that
        # user's stored transcript so the histories visibly stay separate.
        return (
            f"### 👤 Chatting as **{user_id}**",
            load_history(user_id),
            gr.update(placeholder=f"Ask a question as {user_id}..."),
        )

    user_dropdown.change(switch_user, user_dropdown, [header, chatbot, msg])
    msg.submit(respond, [msg, chatbot, user_dropdown], [chatbot, msg])
    send.click(respond, [msg, chatbot, user_dropdown], [chatbot, msg])
    reload_btn.click(load_history, user_dropdown, chatbot)


## 18. Launch the Chat App

Run the cell below to start the Gradio server. A local URL will appear; open it to chat with your RAG app. This brings vector search, hybrid search, semantic cache, and chat history together in one app.

> **Where the chat history comes from.** The transcript is read at runtime from the `chat_history` container, keyed by the `user_id` selected in the profile dropdown, so it isn't hardcoded. Ask James and Saurish different questions, switch between them with the dropdown, then restart the kernel and re-run: the chat repopulates from the database, so the histories persist across the process and stay separate. You can also inspect them with `get_history("James").messages` versus `get_history("Saurish").messages`, or open the container in the Azure portal's Data Explorer to see the per-`user_id` documents.


In [23]:
# Start the Gradio server (non-blocking so the notebook kernel stays free).
# Open the printed local URL in your browser to chat with your RAG app.
demo.launch(prevent_thread_lock=True, height=1000)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 19. Stop the Chat App

When you're done, run this to shut down the Gradio server. We also clear the semantic cache and reset each demo user's chat history here so the notebook starts fresh next time, because otherwise responses cached in this run, which persist in the Cosmos DB `semantic_cache` container, could be reused and tagged **🟢 Cached response** on your next run even for brand-new questions, and the old James and Saurish transcripts (stored in the `chat_history` container) would still load into the chat.


In [ ]:
# Stop local Gradio server
demo.close()

# Also reset the semantic cache so the next run starts clean. The cache is
# persisted in the Cosmos DB `semantic_cache` container, so without this a fresh
# run could reuse answers cached earlier. A quick lookup registers the cache's
# container handle for the current model so clear() can reach the stored entries.
try:
    semantic_cache.lookup("warmup", llm._get_llm_string())
    semantic_cache.clear()
    print("🧹 Semantic cache cleared. The next run starts fresh.")
except Exception as e:
    print(f"Could not clear semantic cache: {e}")

# Reset each demo user's chat history too, since it also persists in Cosmos DB
# (the `chat_history` container). Without this, James's and Saurish's old
# transcripts would reload into the chat on the next run.
for _user in DEMO_USERS:
    try:
        get_history(_user).clear()
        print(f"🧹 Chat history cleared for {_user}.")
    except Exception as e:
        print(f"Could not clear chat history for {_user}: {e}")


Closing server running on port: 7860
🧹 Semantic cache cleared . The next run starts fresh.


## What We Built

You now have a practical LangChain pattern that includes:

- Vector retrieval for semantic relevance
- Hybrid retrieval (vector + full-text) for better precision and recall
- Grounded answer generation with system prompts
- Semantic cache for cost/latency optimization
- Chat history for short-term conversational memory

All powered by Azure Cosmos DB for NoSQL as the single data persistence layer, and Azure OpenAI for embeddings and chat models.

From here, you can tune chunking, prompts, retrieval thresholds, and extend with custom tools based on your domain needs.